# T6 · Build FactShipment
Subtasks: build the six key columns, the degenerate key and the measures; reconcile; answer the three questions. All in this one notebook.

In [1]:
import pandas as pd, numpy as np, os
pd.set_option("display.max_colwidth", None, "display.width", 250, "display.max_rows", 100)

p = next(x for x in ["Shipments_Unified.csv", "../Shipments_Unified.csv", "/mnt/user-data/uploads/Shipments_Unified.csv"] if os.path.exists(x))
src = pd.read_csv(p, parse_dates=["ShipDate"])
assert len(src) == 34 and src.WeightKG.sum() == 578_650 and src.CostPKR.sum() == 4_808_000

# ---------- Dimensions from T4 / T5 (rebuilt compactly so this notebook runs on its own) ----------
city_key = {"Karachi":1,"Gwadar":2,"Lahore":3,"Multan":4,"Islamabad":5,"Quetta":6,"Faisalabad":7,
            "Rawalpindi":8,"Sialkot":9,"Peshawar":10,"Hyderabad":11,"Sukkur":12,"Gujranwala":13,"Abbottabad":14}
alias_city = {"Karachi Port":"Karachi","Port Qasim":"Karachi","Karachi Cantt":"Karachi","Pipri Marshalling Yard":"Karachi","Karachi":"Karachi",
              "Gwadar Port":"Gwadar","Lahore":"Lahore","Lahore Dry Port":"Lahore","Multan":"Multan","Islamabad":"Islamabad",
              "Islamabad Dry Port":"Islamabad","Quetta":"Quetta","Faisalabad":"Faisalabad","Faisalabad Dry Port":"Faisalabad",
              "Rawalpindi":"Rawalpindi","Sialkot Dry Port":"Sialkot","Sialkot":"Sialkot","Peshawar":"Peshawar","Peshawar Cantt":"Peshawar",
              "Hyderabad":"Hyderabad","Sukkur":"Sukkur","Gujranwala":"Gujranwala","Abbottabad":"Abbottabad"}
province = {"Karachi":"Sindh","Hyderabad":"Sindh","Sukkur":"Sindh","Gwadar":"Balochistan","Quetta":"Balochistan",
            "Lahore":"Punjab","Faisalabad":"Punjab","Multan":"Punjab","Sialkot":"Punjab","Gujranwala":"Punjab","Rawalpindi":"Punjab",
            "Islamabad":"Islamabad Capital Territory","Peshawar":"Khyber Pakhtunkhwa","Abbottabad":"Khyber Pakhtunkhwa"}

# DimCity: one row per city + alias table used for the lookup + the -1 row
DimCity = pd.DataFrame([(k, c, province[c]) for c, k in city_key.items()] + [(-1, "Unknown", "Unknown")],
                       columns=["CityKey", "CityName", "Province"])
CityAlias = pd.DataFrame({"RawString": list(alias_city), "CityKey": [city_key[c] for c in alias_city.values()]})

# DimRoute: conform first, then dedupe on the key pair (first-appearance order), plus -1
pairs = pd.DataFrame({"OriginCityKey": src.Origin.map(dict(zip(CityAlias.RawString, CityAlias.CityKey))),
                      "DestCityKey":   src.Destination.map(dict(zip(CityAlias.RawString, CityAlias.CityKey)))}).drop_duplicates().reset_index(drop=True)
DimRoute = pd.concat([pairs.assign(RouteKey=range(1, len(pairs) + 1)),
                      pd.DataFrame([{"RouteKey": -1, "OriginCityKey": -1, "DestCityKey": -1}])], ignore_index=True)

DimMode   = pd.DataFrame([(1, "Marine"), (2, "Rail"), (3, "Road"), (-1, "Unknown")], columns=["ModeKey", "ModeName"])
DimStatus = pd.DataFrame([(1, "Delivered"), (2, "In Transit"), (3, "Delayed"), (-1, "Unknown")], columns=["StatusKey", "StatusName"])
DimDate   = pd.DataFrame({"FullDate": pd.date_range("2026-02-01", "2026-02-28")})
DimDate["DateKey"] = DimDate.FullDate.dt.strftime("%Y%m%d").astype(int)
assert len(DimCity) == 15 and len(DimRoute) == 21 and len(DimDate) == 28

## FactShipment: six keys + one degenerate key + three measures

In [2]:
def build_fact(df):
    """Swap every text column for an integer key by LOOKUP (left join + -1), never typed digits."""
    x = df.copy()
    alias = CityAlias.set_index("RawString").CityKey
    x["OriginCityKey"] = x.Origin.map(alias)                       # NaN when the string is unmapped
    x["DestCityKey"]   = x.Destination.map(alias)
    x = x.merge(DimRoute[DimRoute.RouteKey != -1][["RouteKey", "OriginCityKey", "DestCityKey"]],
                on=["OriginCityKey", "DestCityKey"], how="left")
    x["ModeKey"]   = x.SourceSystem.map(DimMode.set_index("ModeName").ModeKey)
    x["StatusKey"] = x.Status.map(DimStatus.set_index("StatusName").StatusKey)
    x["DateKey"]   = x.ShipDate.dt.strftime("%Y%m%d").astype(int)
    x["DateKey"]   = x.DateKey.where(x.DateKey.isin(DimDate.DateKey))
    keys = ["DateKey", "OriginCityKey", "DestCityKey", "RouteKey", "ModeKey", "StatusKey"]
    x[keys] = x[keys].fillna(-1).astype(int)                       # the IFERROR(..., -1) step
    x["ShipmentCount"] = 1
    return x.rename(columns={"CostPKR": "FreightCostPKR"})[keys + ["ShipmentID", "WeightKG", "FreightCostPKR", "ShipmentCount"]]

FactShipment = build_fact(src)
FactShipment

,DateKey,OriginCityKey,DestCityKey,RouteKey,ModeKey,StatusKey,ShipmentID,WeightKG,FreightCostPKR,ShipmentCount
0,20260201,1,3,1,1,1,MR-1001,18000,142000,1
1,20260202,1,7,2,1,1,MR-1002,22500,171000,1
2,20260203,1,3,1,1,2,MR-1003,15000,119000,1
3,20260204,2,6,3,1,1,MR-1004,20000,188000,1
4,20260205,1,4,4,1,1,MR-1005,16500,128000,1
5,20260206,1,9,5,1,3,MR-1006,19000,154000,1
6,20260207,1,3,1,1,1,MR-1007,14000,109000,1
7,20260209,1,10,6,1,2,MR-1008,21000,197000,1
8,20260210,2,1,7,1,1,MR-1009,17500,98000,1
9,20260211,1,5,8,1,1,MR-1010,15500,139000,1


## Rule and reconciliation checks

In [3]:
keys = ["DateKey", "OriginCityKey", "DestCityKey", "RouteKey", "ModeKey", "StatusKey"]

# Rule: no descriptive text crosses the gap. Only the degenerate ShipmentID may be a string.
text_cols = [c for c in FactShipment.columns if FactShipment[c].dtype == object or str(FactShipment[c].dtype).startswith("str")]
assert text_cols == ["ShipmentID"], text_cols
assert all(pd.api.types.is_integer_dtype(FactShipment[k]) for k in keys)

# Province for T7 checks 9-11, looked up through DimCity (a verification join, NOT a column of the fact)
prov = FactShipment.merge(DimCity[["CityKey", "Province"]], left_on="OriginCityKey", right_on="CityKey").merge(
        DimMode, on="ModeKey")
chk = pd.DataFrame([
 (" 1 Row count of FactShipment",              34,        len(FactShipment)),
 (" 2 Grand total WeightKG",                   578_650,   int(FactShipment.WeightKG.sum())),
 (" 3 Grand total FreightCostPKR",             4_808_000, int(FactShipment.FreightCostPKR.sum())),
 (" 4 Shipment legs (SUM of Count)",           34,        int(FactShipment.ShipmentCount.sum())),
 (" 5 Distinct ShipmentID values",             34,        FactShipment.ShipmentID.nunique()),
 (" 9 WeightKG Sindh / Marine",                169_000,   int(prov[(prov.Province == "Sindh") & (prov.ModeName == "Marine")].WeightKG.sum())),
 ("10 WeightKG Punjab / Road",                 44_100,    int(prov[(prov.Province == "Punjab") & (prov.ModeName == "Road")].WeightKG.sum())),
 ("11 WeightKG Sindh, all modes",              484_350,   int(prov[prov.Province == "Sindh"].WeightKG.sum())),
 ("12 Fact rows pointing at an Unknown key",   0,         int((FactShipment[keys] == -1).any(axis=1).sum())),
 ("13 Cost per KG, grand total (SUM / SUM)",   8.31,      round(FactShipment.FreightCostPKR.sum() / FactShipment.WeightKG.sum(), 2)),
], columns=["T7 check", "Expected", "Result"])
chk["Pass?"] = (chk.Expected == chk.Result).map({True: "pass", False: "FIX"})
assert (chk["Pass?"] == "pass").all()
chk

,T7 check,Expected,Result,Pass?
0,1 Row count of FactShipment,34.00,34.00,pass
1,2 Grand total WeightKG,578650.00,578650.00,pass
2,3 Grand total FreightCostPKR,4808000.00,4808000.00,pass
3,4 Shipment legs (SUM of Count),34.00,34.00,pass
4,5 Distinct ShipmentID values,34.00,34.00,pass
5,9 WeightKG Sindh / Marine,169000.00,169000.00,pass
6,10 WeightKG Punjab / Road,44100.00,44100.00,pass
7,"11 WeightKG Sindh, all modes",484350.00,484350.00,pass
8,12 Fact rows pointing at an Unknown key,0.00,0.00,pass
9,"13 Cost per KG, grand total (SUM / SUM)",8.31,8.31,pass


## Q1 evidence: what IFERROR(..., -1) buys and costs

In [4]:
# Add one row whose Origin is a spelling nobody has mapped
test = pd.concat([src, src.iloc[[0]].assign(ShipmentID="TEST-1", Origin="KHI Port Trust")], ignore_index=True)

with_unknown = build_fact(test)                                           # lookup falls back to -1
inner = test.merge(CityAlias, left_on="Origin", right_on="RawString", how="inner")   # inner join drops it

print("Fallback to -1 : rows =", len(with_unknown), "| rows at Unknown key =", int((with_unknown.OriginCityKey == -1).sum()),
      "| WeightKG =", int(with_unknown.WeightKG.sum()))
print("Inner join     : rows =", len(inner), "(the shipment silently vanished)")

# the cost: the same -1 is produced whatever went wrong, and totals still tie
broken = build_fact(src.assign(Status=src.Status.str.upper()))            # a different bug: status case mismatch
print("A different bug (status text changed): StatusKey = -1 on", int((broken.StatusKey == -1).sum()), "rows; row count and totals still pass:",
      len(broken) == 34 and broken.WeightKG.sum() == 578_650)

Fallback to -1 : rows = 35 | rows at Unknown key = 1 | WeightKG = 596650
Inner join     : rows = 34 (the shipment silently vanished)
A different bug (status text changed): StatusKey = -1 on 34 rows; row count and totals still pass: True


## The three questions

**Q1. What does IFERROR(..., -1) buy you, and what does it cost you?**
*Buys:* the fact row is never lost. An unmapped city, mode or status lands on the `-1` Unknown row, so row count and totals stay at 34 rows / 578,650 kg / PKR 4,808,000, and the problem becomes a number you can count (T7 check #12) instead of a shipment that vanished in an inner join, as the test above shows. *Costs:* it hides the reason. A new spelling, a missing alias, a case mismatch or a broken lookup range all produce the same `-1`, and because the totals still reconcile, checks #1 to #4 stay green while the data is wrong. The fallback is only safe if someone actually reads check #12 and works out which column and which row failed. It also means the lookup can never be trusted merely because it returned a value.

**Q2. Why do the text columns (Origin, Destination, Source, Status) not cross the gap?**
Descriptive text belongs in the dimension, where it is stored once; in the fact it would be repeated on every row (34 times here, millions in production) and would have to be corrected on every row, not in one place. The Karachi problem shows why: five spellings in the fact would mean five separate values to group and filter on, whereas the key resolves all five to `CityKey = 1`. Integer keys are also smaller and faster to join, let one conformed `DimCity` serve other facts (fuel, complaints), and keep the fact at its declared grain: once a text attribute such as a customer name is in the fact, the pressure to duplicate rows to fit it follows (T2 Part C).

**Q3. ShipmentID is in the fact table but joins to nothing. Defend keeping it.**
It is a **degenerate dimension**: the identifier of the transaction itself, with no attributes of its own, so a dimension table would be a one-column table. Keeping it gives: traceability back to the source record and the paper trail (every one of the 34 rows can be found in its source system); a way to verify the grain (T7 check #5, 34 distinct IDs for 34 rows, and a way to spot a duplicated row such as RD-501 in the T2 case); the key that groups legs of one consignment when multi-modal freight arrives; and drill-through from a total down to the individual shipment. Its job is identity, not description, so it needs no join.